# Fraud Shield - Feature Engineering

Formalizes the features explored in `01_eda.ipynb` into reusable transforms
from `src/features/engineering.py` and `src/features/imbalance.py`, then
writes the processed dataset to `data/processed/` (and, later, to
SageMaker Feature Store).

**Inputs:** `data/raw/fraudTrain.csv`, `data/raw/fraudTest.csv`
**Outputs:** `data/processed/train_features.parquet`, `data/processed/test_features.parquet`


In [1]:
import sys, os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)
    
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.features.engineering import (
    add_geo_distance,
    add_transaction_velocity,
    add_spending_profile,
    add_category_amt_zscore,
)
from src.features.imbalance import compute_class_weights

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)


## 1. Load raw data


In [2]:
train = pd.read_csv('../data/raw/fraudTrain.csv')
test = pd.read_csv('../data/raw/fraudTest.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
train.head()


Train shape: (1296675, 23)
Test shape:  (555719, 23)


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


## 2. Add an event-time / transaction id column

Needed for the rolling-window velocity feature, and later required by
SageMaker Feature Store (`record_identifier_name`, `event_time_feature_name`).


In [3]:
def prep_base(df):
    df = df.copy()
    df = df.rename(columns={'trans_date_trans_time': 'datetime'})
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['event_time'] = df['datetime'].astype('int64') // 10**9  # unix seconds, required by Feature Store
    if 'trans_num' in df.columns:
        df['transaction_id'] = df['trans_num']
    else:
        df['transaction_id'] = df.index.astype(str)
    return df

train = prep_base(train)
test = prep_base(test)


## 3. Geo-distance feature

Haversine distance between cardholder location (`lat`/`long`) and
merchant location (`merch_lat`/`merch_long`).


In [4]:
train = add_geo_distance(train)
test = add_geo_distance(test)

train[['lat', 'long', 'merch_lat', 'merch_long', 'geo_distance_km']].head()


,lat,long,merch_lat,merch_long,geo_distance_km
0,36.0788,-81.1781,36.011293,-82.048315,78.597568
1,48.8878,-118.2105,49.159047,-118.186462,30.212176
2,42.1808,-112.2620,43.150704,-112.154481,108.206083
3,46.2306,-112.1138,47.034331,-112.561071,95.673231
4,38.4207,-79.4629,38.674999,-78.632459,77.556744


## 4. Transaction velocity

Rolling count of transactions per card number in a trailing 1-hour window.
This requires the data sorted by `cc_num` and `datetime`, which
`add_transaction_velocity` handles internally.

**Note:** this is O(n log n) per card and can take a few minutes on the
full ~1.3M-row training set.


In [5]:
train = add_transaction_velocity(train, window='1h')
test = add_transaction_velocity(test, window='1h')

train[['cc_num', 'datetime', 'amt', 'txn_velocity']].head(10)


,cc_num,datetime,amt,txn_velocity
0,60416207185,2019-01-01 12:47:15,7.27,1.0
1,60416207185,2019-01-02 08:44:57,52.94,1.0
2,60416207185,2019-01-02 08:47:36,82.08,2.0
3,60416207185,2019-01-02 12:38:14,34.79,1.0
4,60416207185,2019-01-02 13:10:46,27.18,2.0
5,60416207185,2019-01-03 13:56:35,6.87,1.0
6,60416207185,2019-01-03 17:05:10,8.43,1.0
7,60416207185,2019-01-04 13:59:55,117.11,1.0
8,60416207185,2019-01-04 21:17:22,26.74,1.0
9,60416207185,2019-01-05 00:42:24,105.20,1.0


## 5. Spending profile (per-cardholder amount z-score)


In [6]:
train = add_spending_profile(train)
test = add_spending_profile(test)

train[['cc_num', 'amt', 'cc_avg_amt', 'cc_std_amt', 'amt_zscore']].head(10)


,cc_num,amt,cc_avg_amt,cc_std_amt,amt_zscore
0,60416207185,7.27,56.023366,122.632635,-0.397556
1,60416207185,52.94,56.023366,122.632635,-0.025143
2,60416207185,82.08,56.023366,122.632635,0.212477
3,60416207185,34.79,56.023366,122.632635,-0.173146
4,60416207185,27.18,56.023366,122.632635,-0.235201
5,60416207185,6.87,56.023366,122.632635,-0.400818
6,60416207185,8.43,56.023366,122.632635,-0.388097
7,60416207185,117.11,56.023366,122.632635,0.498127
8,60416207185,26.74,56.023366,122.632635,-0.238789
9,60416207185,105.20,56.023366,122.632635,0.401008


## 6. Check for NaN/inf introduced by feature engineering

`amt_zscore` will be NaN when a cardholder has only one transaction
(`cc_std_amt` = 0 or undefined). `txn_velocity` will be NaN for a card's
very first transaction if the rolling window has no prior data.


In [7]:
engineered_cols = ['geo_distance_km', 'txn_velocity', 'cc_avg_amt', 'cc_std_amt', 'amt_zscore']
print('Train NaN counts:')
print(train[engineered_cols].isnull().sum())
print()
print('Test NaN counts:')
print(test[engineered_cols].isnull().sum())


Train NaN counts:
geo_distance_km    0
txn_velocity       0
cc_avg_amt         0
cc_std_amt         0
amt_zscore         0
dtype: int64

Test NaN counts:
geo_distance_km    0
txn_velocity       0
cc_avg_amt         0
cc_std_amt         0
amt_zscore         0
dtype: int64


In [8]:
# Impute: 0 is a reasonable neutral fill for a first-ever transaction
# (no prior velocity, no prior spending profile deviation)
for df in (train, test):
    df['txn_velocity'] = df['txn_velocity'].fillna(1)
    df['amt_zscore'] = df['amt_zscore'].fillna(0)
    df['cc_std_amt'] = df['cc_std_amt'].fillna(0)

print('Remaining NaNs:')
print(train[engineered_cols].isnull().sum().sum(), '(train)')
print(test[engineered_cols].isnull().sum().sum(), '(test)')


Remaining NaNs:
0 (train)
0 (test)


In [9]:
train = add_category_amt_zscore(train)
test = add_category_amt_zscore(test)

train[['category', 'amt', 'cat_avg_amt', 'cat_std_amt', 'amt_category_zscore']].head(10)

,category,amt,cat_avg_amt,cat_std_amt,amt_category_zscore
0,misc_net,7.27,80.865095,166.860495,-0.441058
1,gas_transport,52.94,63.434572,15.886738,-0.660587
2,gas_transport,82.08,63.434572,15.886738,1.173647
3,kids_pets,34.79,57.536871,48.722021,-0.466870
4,home,27.18,58.270139,48.681218,-0.638648
5,shopping_net,6.87,88.424076,247.225566,-0.329877
6,food_dining,8.43,51.086905,48.854164,-0.873148
7,home,117.11,58.270139,48.681218,1.208677
8,personal_care,26.74,47.967678,49.229970,-0.431194
9,grocery_pos,105.20,116.960986,53.214365,-0.221011


## 7. Encode categorical features

`category` (merchant category) and `gender` are low-cardinality and
one-hot encode cleanly. High-cardinality fields (`merchant`, `job`, `city`,
`state`) are left out of the baseline feature set for now -- worth revisiting
as target-encoded features if the baseline models need more signal.


In [10]:
categorical_cols = ['category', 'gender']

train_encoded = pd.get_dummies(train, columns=categorical_cols, prefix=categorical_cols)
test_encoded = pd.get_dummies(test, columns=categorical_cols, prefix=categorical_cols)

# Align columns in case a category appears in one split but not the other
train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

print(f'Train encoded shape: {train_encoded.shape}')
print(f'Test encoded shape:  {test_encoded.shape}')


Train encoded shape: (1296675, 47)
Test encoded shape:  (555719, 47)


## 8. Final feature set

Select the columns that will feed the models: engineered features plus
the encoded categoricals and core numeric fields.


In [17]:
feature_cols = ['amt', 'city_pop', 'geo_distance_km', 'txn_velocity', 'amt_zscore', 'amt_category_zscore']
feature_cols = feature_cols + [c for c in train_encoded.columns if c.startswith('category_') or c.startswith('gender_')]

id_cols = ['transaction_id', 'event_time', 'cc_num']
target_col = 'is_fraud'

train_final = train_encoded[id_cols + feature_cols + [target_col]].copy()
test_final = test_encoded[id_cols + feature_cols + [target_col]].copy()

print(f'Final feature count: {len(feature_cols)}')
print('amt_category_zscore' in feature_cols)
train_final.head()

Final feature count: 22
True


,transaction_id,event_time,cc_num,amt,city_pop,geo_distance_km,txn_velocity,amt_zscore,amt_category_zscore,category_entertainment,category_food_dining,category_gas_transport,category_grocery_net,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel,gender_F,gender_M,is_fraud
0,98e3dcf98101146a577f85a34e58feec,1546346835,60416207185,7.27,1645,127.606239,1.0,-0.397556,-0.441058,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,0
1,498120fc45d277f7c88e3dba79c33865,1546418697,60416207185,52.94,1645,110.308921,1.0,-0.025143,-0.660587,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,0
2,95f514bb993151347c7acdf8505c3d62,1546418856,60416207185,82.08,1645,21.787261,2.0,0.212477,1.173647,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,0
3,4f0c1a14e0aa7eb56a490780ef9268c5,1546432694,60416207185,34.79,1645,87.204215,1.0,-0.173146,-0.466870,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,0
4,3b2ebd3af508afba959640893e1e82bc,1546434646,60416207185,27.18,1645,74.212965,2.0,-0.235201,-0.638648,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,0


## 9. Class imbalance preview

Compute class weights for cost-sensitive training (used directly by
`scale_pos_weight` in XGBoost, `class_weight` in sklearn/PyTorch).
SMOTE/undersampling (`src/features/imbalance.py`) will be applied inside the
training notebook, fit only on the training fold to avoid leakage.


In [18]:
weights = compute_class_weights(train_final[target_col])
print('Class weights:', weights)

fraud_rate = train_final[target_col].mean() * 100
print(f'Fraud rate: {fraud_rate:.4f}%')


Class weights: {0: 0.5029111776656126, 1: 86.37589928057554}
Fraud rate: 0.5789%


## 10. Save processed features


In [19]:
train_final.to_parquet('../data/processed/train_features.parquet', index=False)
test_final.to_parquet('../data/processed/test_features.parquet', index=False)

print('Saved train_features.parquet and test_features.parquet to data/processed/')


Saved train_features.parquet and test_features.parquet to data/processed/


In [20]:
print(feature_cols)

['amt', 'city_pop', 'geo_distance_km', 'txn_velocity', 'amt_zscore', 'amt_category_zscore', 'category_entertainment', 'category_food_dining', 'category_gas_transport', 'category_grocery_net', 'category_grocery_pos', 'category_health_fitness', 'category_home', 'category_kids_pets', 'category_misc_net', 'category_misc_pos', 'category_personal_care', 'category_shopping_net', 'category_shopping_pos', 'category_travel', 'gender_F', 'gender_M']


In [21]:
check = pd.read_parquet('../data/processed/train_features.parquet')
print(check.shape)
print('amt_category_zscore' in check.columns)

(1296675, 26)
True


## 11. (Later) Push to SageMaker Feature Store

Once AWS credentials and `SAGEMAKER_ROLE_ARN` are configured (see `.env`),
ingest the engineered features so both training and the real-time inference
endpoint read from the same online/offline store.


In [15]:
# from src.features.feature_store import get_or_create_feature_group, ingest
#
# feature_group = get_or_create_feature_group(train_final)
# ingest(feature_group, train_final)


## 12. Next steps

- Confirm feature distributions look sane post-imputation (rerun EDA-style
  plots on `train_final` if anything looks off)
- Move to `03_baseline_models.ipynb`: apply SMOTE/undersampling on the
  training fold only, then train LogReg / Random Forest / XGBoost
- Revisit high-cardinality categoricals (`merchant`, `job`, `state`) as
  target-encoded features if baseline recall is weak
